# Administração de Redes: Trabalho 2
## Montagem do laboratório OSPF no GNS3 (2 AS, 3 áreas)

**Mestrado em Engenharia de Redes, FCUP, 2026/27**

Este documento regista **como o laboratório foi montado e validado**: plano de endereçamento, configuração de cada router, comandos de verificação e os erros encontrados pelo caminho (com a causa e a correção). As respostas às questões 1 a 7 do enunciado ficam para uma fase posterior, com as capturas feitas sobre esta rede.

> **Estado atual (2026-10-15):** rede convergida. Ping e traceroute entre Terminal 1 e Terminal 2 funcionam nos dois sentidos, e todos os routers têm a configuração gravada (`write memory`).

## 0. Topologia, áreas e plano de endereçamento

Duas redes autónomas ligadas pelo R2:

- **AS1** (área 0): R1 e R2.
- **AS2**: R2 (lado AS2), R3, R4, R5 e R6 na área 0; R6, R7 e R9 na área 1; R6, R8 e R9 na área 2.
- **R6** é o ABR das áreas 0, 1 e 2. **R9** liga as áreas 1 e 2 e não tem ligação direta à área 0. **R2** é ASBR e corre dois processos OSPF.

### Ligações e endereços (portas reais do projeto GNS3)

| Router | Porta | Liga a | Endereço | Rede | Área |
|---|---|---|---|---|---|
| R1 | Fa1/1 | R2 | 192.168.1.129/26 | 192.168.1.128/26 | AS1, 0 |
| R1 | Fa1/0 | Terminal 1 | 192.168.1.1/25 | 192.168.1.0/25 | AS1, 0 |
| R2 | Fa1/1 | R1 | 192.168.1.130/26 | 192.168.1.128/26 | AS1, 0 |
| R2 | Fa1/0 | C1 | 192.168.1.193/26 | 192.168.1.192/26 | AS1, 0 |
| R2 | Fa2/0 | R3 | 192.168.50.1/24 | 192.168.50.0/24 | AS2, 0 |
| R3 | Fa1/1 | R2 | 192.168.50.2/24 | 192.168.50.0/24 | AS2, 0 |
| R3 | Fa1/0 | Nuvem | 192.168.60.1/24 | 192.168.60.0/24 | AS2, 0 |
| R3 | Fa2/0 | Switch | 192.168.100.1/24 | 192.168.100.0/24 | AS2, 0 |
| R4 | Fa1/0 | Switch | 192.168.100.253/24 | 192.168.100.0/24 | AS2, 0 |
| R5 | Fa1/0 | Switch | 192.168.100.254/24 | 192.168.100.0/24 | AS2, 0 |
| R6 | Fa1/0 | Switch | 192.168.100.2/24 | 192.168.100.0/24 | AS2, 0 |
| R6 | Fa1/1 | R7 | 172.20.1.1/30 | 172.20.1.0/30 | AS2, 1 |
| R6 | Se2/0 | R8 | 172.20.1.14/30 | 172.20.1.12/30 | AS2, 2 |
| R7 | Fa1/0 | R6 | 172.20.1.2/30 | 172.20.1.0/30 | AS2, 1 |
| R7 | Fa1/1 | R9 | 172.20.1.5/30 | 172.20.1.4/30 | AS2, 1 |
| R7 | Fa2/0 | Terminal 2 | 172.16.1.1/24 | 172.16.1.0/24 | AS2, 1 |
| R8 | Se2/0 | R6 | 172.20.1.13/30 | 172.20.1.12/30 | AS2, 2 |
| R8 | Fa1/1 | R9 | 172.20.1.10/30 | 172.20.1.8/30 | AS2, 2 |
| R8 | Fa1/0 | C3 | 172.18.1.1/24 | 172.18.1.0/24 | AS2, 2 |
| R9 | Fa1/0 | R7 | 172.20.1.6/30 | 172.20.1.4/30 | AS2, 1 |
| R9 | Fa1/1 | R8 | 172.20.1.9/30 | 172.20.1.8/30 | AS2, 2 |
| R9 | Fa2/0 | C4 | 172.17.0.1/16 | 172.17.0.0/16 | AS2, 2 |

**Terminais (VPCS)**

| Terminal | Endereço | Gateway |
|---|---|---|
| Terminal 1 | 192.168.1.123/25 | 192.168.1.1 |
| Terminal 2 | 172.16.1.2/24 | 172.16.1.1 |

**Router IDs** (regra do enunciado): `1.0.0.n` no AS1 e `2.0.0.n` no AS2. O R2 tem dois: `1.0.0.2` (processo 1, AS1) e `2.0.0.2` (processo 2, AS2).

![Planta da rede do enunciado (AS1, AS2 e áreas OSPF)](evidencias/t2_00_topologia.png)

*Planta da rede do enunciado (AS1, AS2 e áreas OSPF)*


## 1. Como se calculam máscaras e wildcards

Os prefixos `/25`, `/26`, `/24` e `/16` vêm escritos nas nuvens da figura. Os das ligações `172.20.1.x` e `192.168.100.x` não vêm, e foram **inferidos**:

- **172.20.1.x: /30.** Cada ligação tem só dois routers, e os pares de endereços ficam em blocos consecutivos de 4 (.0-.3, .4-.7, .8-.11, .12-.15).
- **192.168.100.0: /24.** Os endereços .1, .2, .253 e .254 encaixam naturalmente.

Se a máscara diferir entre dois vizinhos, a adjacência fica presa em `INIT` ou `EXSTART`.

**Método (exemplo /26)**

1. Bits de host = 32 menos o prefixo = 6.
2. Tamanho do bloco = 2^6 = 64.
3. Máscara no último octeto = 256 menos o bloco = 192, logo `255.255.255.192`.
4. **Wildcard** (inverso da máscara) = 255 menos cada octeto = bloco menos 1 = `0.0.0.63`.

| Prefixo | Máscara | Wildcard | Bloco |
|---|---|---|---|
| /16 | 255.255.0.0 | 0.0.255.255 | 65536 |
| /24 | 255.255.255.0 | 0.0.0.255 | 256 |
| /25 | 255.255.255.128 | 0.0.0.127 | 128 |
| /26 | 255.255.255.192 | 0.0.0.63 | 64 |
| /30 | 255.255.255.252 | 0.0.0.3 | 4 |

**Para achar o endereço de rede** (o que o comando `network` pede), localize o endereço no bloco: com /30, 172.20.1.5 e 172.20.1.6 caem em .4-.7, logo a rede é `172.20.1.4`. Na wildcard, `0` significa "este bit tem de coincidir" e `1` significa "ignorar".

Os três blocos do AS1 não se sobrepõem e preenchem 192.168.1.0/24: `.0/25` (.0-.127), `.128/26` (.128-.191) e `.192/26` (.192-.255). É por isso que o enunciado manda sumarizá-los em `192.168.1.0/24`.

## 2. Regras transversais da configuração

- **`auto-cost reference-bandwidth 1000`** em todos os routers (e nos dois processos do R2). Com isto cada FastEthernet custa 10 (1000/100 Mbps), como o enunciado pede.
- **`passive-interface default`** seguido de `no passive-interface <if>` só nas interfaces que ligam a outros routers. As LANs com terminais ou nuvens continuam anunciadas pelo `network`, mas não enviam hellos OSPF.
- **`router-id`** fixado à mão em cada processo.
- **Número do processo OSPF:** é local ao router. Usou-se 1 para o AS1 e 2 para o AS2, também no R2.
- **`no logging console`** durante a configuração, para as mensagens `%OSPF-4-ERRRCV` não interromperem a escrita (ver erro 2 na secção 5).
- **`write memory`** em todos os routers no fim. Sem gravar, a configuração perde-se ao parar o router no GNS3.

Convenção dos blocos: a linha `network` define em que área fica cada interface e a linha `no passive-interface` define se envia hellos. **São independentes, e as duas têm de existir.**

## 3. Configuração por router (confRes)

Cada bloco parte do modo EXEC privilegiado. Se o prompt for `Router>`, escreve-se primeiro `enable` e `configure terminal`.

### 3.1 R1 (AS1)

```
enable
configure terminal
hostname R1
no ip domain-lookup
interface FastEthernet1/1
 description Link to R2
 ip address 192.168.1.129 255.255.255.192
 no shutdown
 exit
interface FastEthernet1/0
 description LAN Terminal 1
 ip address 192.168.1.1 255.255.255.128
 no shutdown
 exit
router ospf 1
 router-id 1.0.0.1
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/1
 network 192.168.1.128 0.0.0.63 area 0
 network 192.168.1.0 0.0.0.127 area 0
 end
write memory
```

**Terminal 1 (VPCS):** `ip 192.168.1.123/25 192.168.1.1` e `save`.

### 3.2 R2 (ASBR, dois processos, redistribuição mútua e sumarização)

```
enable
configure terminal
hostname R2
no ip domain-lookup
interface FastEthernet1/1
 description Link to R1
 ip address 192.168.1.130 255.255.255.192
 no shutdown
 exit
interface FastEthernet1/0
 description LAN C1
 ip address 192.168.1.193 255.255.255.192
 no shutdown
 exit
interface FastEthernet2/0
 description Link to R3
 ip address 192.168.50.1 255.255.255.0
 no shutdown
 exit
router ospf 1
 router-id 1.0.0.2
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/1
 network 192.168.1.128 0.0.0.63 area 0
 network 192.168.1.192 0.0.0.63 area 0
 redistribute ospf 2 metric 100 subnets
 exit
router ospf 2
 router-id 2.0.0.2
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet2/0
 network 192.168.50.0 0.0.0.255 area 0
 redistribute ospf 1 metric 100 subnets
 summary-address 192.168.1.0 255.255.255.0
 end
write memory
```

- `redistribute ... metric 100` cumpre "rotas importadas com métrica 100". `subnets` é obrigatório, senão só se redistribuem redes classful.
- `summary-address 192.168.1.0 255.255.255.0` sumariza as redes 192.168.1.x **apenas ao serem injetadas no AS2**. Dentro do AS1 mantêm as máscaras originais. O IOS cria automaticamente uma rota de descarte `192.168.1.0/24 ... Null0`.

### 3.3 R3

```
enable
configure terminal
hostname R3
no ip domain-lookup
interface FastEthernet1/1
 description Link to R2
 ip address 192.168.50.2 255.255.255.0
 no shutdown
 exit
interface FastEthernet1/0
 description LAN 192.168.60.0
 ip address 192.168.60.1 255.255.255.0
 no shutdown
 exit
interface FastEthernet2/0
 description Link to switch
 ip address 192.168.100.1 255.255.255.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.3
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/1
 no passive-interface FastEthernet2/0
 network 192.168.50.0 0.0.0.255 area 0
 network 192.168.60.0 0.0.0.255 area 0
 network 192.168.100.0 0.0.0.255 area 0
 end
write memory
```

### 3.4 R4 e R5 (iguais, mudam endereço, router ID e hostname)

R4 usa `192.168.100.253` e `router-id 2.0.0.4`. R5 usa `192.168.100.254` e `router-id 2.0.0.5`.

```
enable
configure terminal
hostname R4
no ip domain-lookup
interface FastEthernet1/0
 description Link to switch
 ip address 192.168.100.253 255.255.255.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.4
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/0
 network 192.168.100.0 0.0.0.255 area 0
 end
write memory
```

### 3.5 R6 (ABR das áreas 0, 1 e 2)

```
enable
configure terminal
no logging console
hostname R6
no ip domain-lookup
interface FastEthernet1/0
 description Link to switch
 ip address 192.168.100.2 255.255.255.0
 no shutdown
 exit
interface FastEthernet1/1
 description Link to R7
 ip address 172.20.1.1 255.255.255.252
 no shutdown
 exit
interface Serial2/0
 description Link to R8
 ip address 172.20.1.14 255.255.255.252
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.6
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/0
 no passive-interface FastEthernet1/1
 no passive-interface Serial2/0
 network 192.168.100.0 0.0.0.255 area 0
 network 172.20.1.0 0.0.0.3 area 1
 network 172.20.1.12 0.0.0.3 area 2
 end
write memory
```

A interface série ficou com custo 647 (largura de banda por omissão de 1544 kbps contra a referência de 1000 Mbps). É o esperado, porque o enunciado só fixa o custo das FastEthernet. Não foi preciso `clock rate`.

### 3.6 R7 (área 1)

```
enable
configure terminal
no logging console
hostname R7
no ip domain-lookup
interface FastEthernet1/0
 description Link to R6
 ip address 172.20.1.2 255.255.255.252
 no shutdown
 exit
interface FastEthernet1/1
 description Link to R9
 ip address 172.20.1.5 255.255.255.252
 no shutdown
 exit
interface FastEthernet2/0
 description LAN Terminal 2
 ip address 172.16.1.1 255.255.255.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.7
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/0
 no passive-interface FastEthernet1/1
 network 172.20.1.0 0.0.0.3 area 1
 network 172.20.1.4 0.0.0.3 area 1
 network 172.16.1.0 0.0.0.255 area 1
 end
write memory
```

**Terminal 2 (VPCS):** `ip 172.16.1.2/24 172.16.1.1` e `save`.

### 3.7 R8 (área 2)

```
enable
configure terminal
no logging console
hostname R8
no ip domain-lookup
interface Serial2/0
 description Link to R6
 ip address 172.20.1.13 255.255.255.252
 no shutdown
 exit
interface FastEthernet1/1
 description Link to R9
 ip address 172.20.1.10 255.255.255.252
 no shutdown
 exit
interface FastEthernet1/0
 description LAN C3
 ip address 172.18.1.1 255.255.255.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.8
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface Serial2/0
 no passive-interface FastEthernet1/1
 network 172.20.1.12 0.0.0.3 area 2
 network 172.20.1.8 0.0.0.3 area 2
 network 172.18.1.0 0.0.0.255 area 2
 end
write memory
```

### 3.8 R9 (áreas 1 e 2, sem ligação direta à área 0)

```
enable
configure terminal
no logging console
hostname R9
no ip domain-lookup
interface FastEthernet1/0
 description Link to R7
 ip address 172.20.1.6 255.255.255.252
 no shutdown
 exit
interface FastEthernet1/1
 description Link to R8
 ip address 172.20.1.9 255.255.255.252
 no shutdown
 exit
interface FastEthernet2/0
 description LAN C4
 ip address 172.17.0.1 255.255.0.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.9
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/0
 no passive-interface FastEthernet1/1
 network 172.20.1.4 0.0.0.3 area 1
 network 172.20.1.8 0.0.0.3 area 2
 network 172.17.0.0 0.0.255.255 area 2
 end
write memory
```

## 4. Comandos de verificação (outRes)

Em cada router, depois de configurar:

```
show ip interface brief
show ip ospf interface brief
show ip ospf neighbor
show ip route
show running-config | section ospf
```

**O que olhar**

- `show ip interface brief`: todas as interfaces em uso `up/up`. O rótulo `SLARP` na coluna Method de uma interface série é apenas a marca do IOS para HDLC. A interface tem o endereço configurado à mão.
- `show ip ospf interface brief`: a coluna **Area** tem de bater com a planta (R6: Fa1/0 na 0, Fa1/1 na 1, Se2/0 na 2) e o custo tem de ser 10 nas FastEthernet.
- `show ip ospf neighbor`: o **Neighbor ID** tem de ser o router ID configurado (`2.0.0.n`), não um endereço de interface.
- `show ip ospf interface <if>`: se aparecer **`No Hellos (Passive interface)`**, a interface está passiva e não forma adjacência.

**Resultados observados no laboratório**

| Verificação | Resultado |
|---|---|
| R2, `show ip ospf` | Dois processos (`ospf 1` ID 1.0.0.2 e `ospf 2` ID 2.0.0.2), "autonomous system boundary router", redistribuição de `ospf 2` com métrica 100 e `subnets` |
| R2, `show ip route` | `192.168.1.0/25` via 192.168.1.129 `[110/20]`, e `192.168.1.0/24 is a summary ... Null0` |
| R3, `show ip route` | `O E2 192.168.1.0/24 [110/100] via 192.168.50.1`: a sumarização chega ao AS2 |
| R5, vizinhos | R3 `FULL/DR` e R4 `FULL/BDR` |
| R6, vizinhos | 2.0.0.3, 2.0.0.4 e 2.0.0.5 na Fa1/0 (todos `FULL`, com os papéis DR/BDR/DROTHER a depender da eleição), 2.0.0.7 na Fa1/1 (`FULL/BDR`) e 2.0.0.8 na Se2/0 (ponto a ponto, sem DR) |
| R7, vizinhos | 2.0.0.9 `FULL/DR` (Fa1/1) e 2.0.0.6 `FULL/DR` (Fa1/0) |
| R1, `show ip route` | Redes do AS2 como `O E2 [110/100]` via 192.168.1.130, e `192.168.1.0/24` também como `O E2` (ver nota na secção 6) |

**Teste de ponta a ponta**

- R1: `ping 172.16.1.2` com 80% de sucesso (o 1.º pacote perde-se na resolução ARP) e `trace 172.16.1.2` com o caminho 192.168.1.130 (R2), 192.168.50.2 (R3), 192.168.100.2 (R6), 172.20.1.2 (R7), 172.16.1.2.
- Terminal 1 (VPCS): `ping 172.16.1.2` com TTL 59 e `trace 172.16.1.2` com o mesmo caminho. A última linha, `Destination port unreachable`, é a forma como o VPCS termina o trace.

**Capturas de ecrã (outRes)**

![R1: configuração OSPF, interfaces na área 0 e running-config](evidencias/t2_r1_configuracao.png)

*R1: configuração OSPF, interfaces na área 0 e running-config*

![R2: interfaces up/up e `show ip ospf` com os dois processos](evidencias/t2_r2_interfaces.png)

*R2: interfaces up/up e `show ip ospf` com os dois processos*

![R2: processo `ospf 1` (ID 1.0.0.2), ASBR e redistribuição com métrica 100](evidencias/t2_r2_ospf_processos.png)

*R2: processo `ospf 1` (ID 1.0.0.2), ASBR e redistribuição com métrica 100*

![R2: área 0 do processo 2 com 2 interfaces](evidencias/t2_r2_ospf_area0.png)

*R2: área 0 do processo 2 com 2 interfaces*

![R2: vizinho R1 em FULL e rota sumarizada 192.168.1.0/24 para Null0](evidencias/t2_r2_vizinhos_rotas.png)

*R2: vizinho R1 em FULL e rota sumarizada 192.168.1.0/24 para Null0*

![R3: vizinho R2 em FULL e `O E2 192.168.1.0/24 [110/100]`](evidencias/t2_r3_vizinhos_rotas.png)

*R3: vizinho R2 em FULL e `O E2 192.168.1.0/24 [110/100]`*

![R4: interface Fa1/0 e vizinho R3](evidencias/t2_r4_vizinhos.png)

*R4: interface Fa1/0 e vizinho R3*

![R5: vizinhos R3 (DR) e R4 (BDR)](evidencias/t2_r5_vizinhos.png)

*R5: vizinhos R3 (DR) e R4 (BDR)*

![R6: network lines corretas, Fa1/0 na área 0, Fa1/1 na 1 e Se2/0 na 2](evidencias/t2_r6_config_corrigida.png)

*R6: network lines corretas, Fa1/0 na área 0, Fa1/1 na 1 e Se2/0 na 2*

![R7: interfaces na área 1 e vizinhos R9 e R6 em FULL](evidencias/t2_r7_vizinhos.png)

*R7: interfaces na área 1 e vizinhos R9 e R6 em FULL*

![R9: Fa1/0 na área 1, Fa1/1 e Fa2/0 na área 2](evidencias/t2_r9_interfaces.png)

*R9: Fa1/0 na área 1, Fa1/1 e Fa2/0 na área 2*

![R1: rotas O E2 do AS2, ping e trace para 172.16.1.2](evidencias/t2_r1_rotas_ping_trace.png)

*R1: rotas O E2 do AS2, ping e trace para 172.16.1.2*

![Terminal 1 (VPCS): ping e trace para 172.16.1.2](evidencias/t2_term1_ping_trace.png)

*Terminal 1 (VPCS): ping e trace para 172.16.1.2*


## 5. Erros encontrados e correções

### Erro 1: comandos rejeitados com `% Invalid input detected`

- **Sintoma:** `hostname R1` e `no ip domain lookup` devolveram `Invalid input` no prompt `Router>`.
- **Causa:** o prompt `Router>` é o modo EXEC de utilizador, onde os comandos de configuração não existem. Além disso, o comando é `no ip domain-lookup`, com hífen.
- **Correção:** `enable`, depois `configure terminal`, e só então os comandos de configuração. O prompt passa a `Router(config)#`.

![Erro 1: `Invalid input` no prompt `Router>`](evidencias/t2_erro1_modo_exec.png)

*Erro 1: `Invalid input` no prompt `Router>`*

### Erro 2: R6 com Fa1/0 na área errada e consola inutilizável

- **Sintoma:** a consola do R6 enchia-se de
  `%OSPF-4-ERRRCV: Received invalid packet: mismatch area ID, from backbone area must be virtual-link but not found from 192.168.100.x`,
  e não era possível escrever comandos.
- **Causa:** o `show running-config | section ospf` mostrava `network 192.168.100.0 0.0.0.255 area 1`. A interface Fa1/0 estava na **área 1**, enquanto R3, R4 e R5 enviam hellos da **área 0**. O R6 rejeitava-os por incompatibilidade de área.
- **Correção:**
  ```
  no logging console
  router ospf 2
   no network 192.168.100.0 0.0.0.255 area 1
   network 192.168.100.0 0.0.0.255 area 0
   network 172.20.1.0 0.0.0.3 area 1
   network 172.20.1.12 0.0.0.3 area 2
  ```
- **Lição:** o `no logging console` impede que as mensagens interrompam a escrita (os caracteres não se perdiam, o log só os intercalava). A linha errada tinha de ser removida com `no network ...`, porque o primeiro `network` que coincide com a interface decide a área.

![Erro 2: mensagens `mismatch area ID` na consola do R6](evidencias/t2_erro2_r6_mismatch_area_log.png)

*Erro 2: mensagens `mismatch area ID` na consola do R6*

![Erro 2: `network 192.168.100.0 0.0.0.255 area 1` e Fa1/0 na área 1](evidencias/t2_erro2_r6_config_area_errada.png)

*Erro 2: `network 192.168.100.0 0.0.0.255 area 1` e Fa1/0 na área 1*

### Erro 3: R5 aparecia como `192.168.100.254` em vez de `2.0.0.5`

- **Sintoma:** no `show ip ospf neighbor` do R6, o vizinho tinha o ID `192.168.100.254`.
- **Causa:** o `router-id 2.0.0.5` não estava em vigor. O IOS só aplica uma mudança de router ID depois de reiniciar o processo, e na falta dela usa o endereço mais alto de uma interface.
- **Correção no R5:**
  ```
  configure terminal
  router ospf 2
   router-id 2.0.0.5
   end
  clear ip ospf process
  write memory
  ```
  Respondeu-se `yes` à confirmação. A entrada antiga `192.168.100.254` desaparece do R6 em menos de um minuto, quando o dead timer expira.
- **Importância:** o enunciado exige router IDs `2.0.0.n` e as questões sobre LSAs olham para o campo *Advertising Router*.

![Erro 3: R6 vê o R5 como 192.168.100.254](evidencias/t2_erro3_r6_vizinhos_id_r5.png)

*Erro 3: R6 vê o R5 como 192.168.100.254*

![Erro 3: após o `clear ip ospf process`, o R5 aparece como 2.0.0.5 (a entrada antiga expira)](evidencias/t2_erro3_r6_vizinhos_apos_fix_r5.png)

*Erro 3: após o `clear ip ospf process`, o R5 aparece como 2.0.0.5 (a entrada antiga expira)*

### Erro 4: R8 sem vizinhos, embora os pings funcionassem

- **Sintoma:** R8 sem qualquer vizinho OSPF, e R6 e R9 sem ver o R8. Os pings entre R8, R9 e R6 tinham sucesso, logo as camadas 1 a 3 estavam corretas.
- **Causa:** `show ip ospf interface FastEthernet1/1` e `show ip ospf interface Serial2/0` mostravam **`No Hellos (Passive interface)`**. O `passive-interface default` estava ativo e faltavam as linhas `no passive-interface` das duas interfaces ligadas a routers.
- **Correção:**
  ```
  undebug all
  configure terminal
  router ospf 2
   no passive-interface Serial2/0
   no passive-interface FastEthernet1/1
   end
  write memory
  ```
  A Fa1/0 (nuvem 172.18.1.0/24) fica passiva.
- **Lição de diagnóstico:** separar primeiro o problema de IP (`ping` aos vizinhos) do problema de OSPF (`show ip ospf interface`, `debug ip ospf hello`). Uma interface passiva mantém o endereço anunciado, mas não envia hellos, por isso o ping funciona e a adjacência não.

![Erro 4: R8 sem vizinhos OSPF](evidencias/t2_erro4_r8_sem_vizinhos.png)

*Erro 4: R8 sem vizinhos OSPF*

![Erro 4: R9 só vê o R7, não vê o R8](evidencias/t2_erro4_r9_vizinhos.png)

*Erro 4: R9 só vê o R7, não vê o R8*

![Erro 4: `No Hellos (Passive interface)` na Fa1/1 e na Se2/0 do R8](evidencias/t2_erro4_r8_passive_hello.png)

*Erro 4: `No Hellos (Passive interface)` na Fa1/1 e na Se2/0 do R8*

### Diagnóstico enganador (falso alarme): `SLARP` na Se2/0 do R8

Por momentos suspeitou-se que o endereço série não estava aplicado. O `show running-config interface Serial2/0` mostrou `ip address 172.20.1.13 255.255.255.252`, ou seja, estava configurado. O rótulo `SLARP` é só a marca do IOS para HDLC, e a causa real era a interface passiva (erro 4).

![Falso alarme: endereço da Se2/0 configurado à mão, rotulado `SLARP`](evidencias/t2_erro4_r8_serial_slarp.png)

*Falso alarme: endereço da Se2/0 configurado à mão, rotulado `SLARP`*

### Boa prática final

Em cada router, comparar `show ip ospf interface brief` e `show running-config | section ospf` com o plano da secção 3. Os erros 2, 3 e 4 seriam todos detetados por esta verificação.

## 6. Observações sobre o comportamento esperado

- **`O E2 192.168.1.0/24` no R1:** o AS2 reanuncia a sumarização do R2 de volta para o AS1, porque a redistribuição é feita nos dois sentidos. É inofensivo: as rotas específicas (`/25` ligada e `/26` aprendidas) ganham por serem mais específicas. O enunciado não pede filtragem.
- **Rotas externas E2 com métrica 100:** a métrica de uma rota E2 não aumenta ao atravessar o AS, por isso aparece 100 em todos os routers.
- **Rede multi-acesso 192.168.100.0/24:** só o DR e o BDR ficam `FULL` com todos. Os restantes ficam em `2WAY` entre si (`DROTH`). Esta observação é relevante para a questão 5 do enunciado.
- **R9 sem ligação à área 0:** o tráfego entre as áreas 1 e 2 e a área 0 passa pelo R6. Este desenho é intencional, e explica a assimetria de caminhos pedida em algumas questões. Não deve ser "corrigido".
- **Ping inicial perdido (`.!!!!`):** é a resolução ARP do primeiro pacote.

## 7. Pendente: roteiro das capturas para as questões 1 a 7

Ainda não respondido. Ordem de arranque sugerida para as capturas:

1. Gravar a configuração em todos os routers (`write memory`) e pará-los.
2. Arrancar R3 a R9 e esperar alguns minutos.
3. Iniciar a captura na ligação R2 a R3 (botão direito na ligação, *Start capture*).
4. Arrancar R2 e aguardar a passagem de `LOADING` a `FULL`. Parar a captura.
5. Arrancar R1 e esperar pela convergência.

Para cortar a ligação R6 a R7 (questão 6), usar `shutdown` na Fa1/1 do R6 e repor com `no shutdown`.

> **Evidências:** as capturas de ecrã estão em `evidencias/` com o prefixo `t2_`. Não há ficheiros `.pcapng` deste trabalho, porque as capturas de pacotes das questões 1 a 7 ainda não foram feitas.
